# Group Project (Part 1 + Part 2): DQR and DQP

This notebook prepares:
1. **Part 1**: Data Quality Report (DQR) for `ppr-group-22312913-train.csv`
2. **Part 2**: Data Quality Plan (DQP) and implementation of cleaning decisions

The workflow and format follows Week 3/4 labs: setup, initial checks, feature types, tables, visualisations, plan, implementation, and validation.


In [ ]:
#install packages before running this script:
%pip install numpy pandas matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 25.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 72.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 23.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 10.5 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 77.2 MB/s  0:00:00
   ━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  2/10 [numpy]  WARNING: The scripts f2py and numpy-config are installed in '/Library/Frameworks/Python.framework/Versions/3.14/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━  4/10 [fonttools]  WARNING: The scripts fonttools, pyftmerge, pyftsubset and ttx are installed in '/Library/Frameworks/Python.framework/Versions/3.14/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to

In [4]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.backends.backend_pdf import PdfPages

try:
    import seaborn as sns
    sns.set_style('whitegrid')
    USE_SEABORN = True
except Exception:
    USE_SEABORN = False

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', '{:,.2f}'.format)


## Step 1: Load Datasets and Basic Inspection
Use training data for DQR/DQP. Keep test data aside for later project parts.


In [26]:
# Load the training data
train_path = 'ppr-group-22312913-train.csv'
df_train = pd.read_csv(train_path)

print('Train shape:', df_train.shape)
df_train.head(5)

Train shape: (54000, 9)


,Date of Sale (dd/mm/yyyy),Address,County,Eircode,Price (€),Not Full Market Price,VAT Exclusive,Description of Property,Property Size Description
0,01/04/2016,"7 MARLTON DEMESNE, DEMESNE, WICKLOW",Wicklow,NaN,"€450,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN
1,15/09/2016,"90 SLIEVE RUA DR, KILMACUD, BLACKROCK",Dublin,NaN,"€560,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN
2,21/12/2016,"APT. 1 ABOVE BERGERACS, MAIN STREET, ROASREA",Tipperary,NaN,"€44,200.00",No,No,Second-Hand Dwelling house /Apartment,NaN
3,17/08/2016,"6 MARIEMOUNT GROVE, BALLINAMORE, COUNTY LEITRIM",Leitrim,NaN,"€65,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN
4,19/04/2016,"THE GLADE, TYMULLEN, MONASTERBOICE",Louth,NaN,"€400,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN


## Step 2: Initial Data Checks
Following the lab pattern: dimensions, dtypes, duplicates, constants, and missingness.


In [6]:
# Check duplicate rows
duplicate_rows = df_train.duplicated().sum()
print(f'Number of duplicate rows: {duplicate_rows}')

# Check for duplicate columns
duplicate_cols = df_train.T.duplicated().sum()
print(f'Number of duplicate columns: {duplicate_cols}')

# Check for missing values
missing_values = df_train.isnull().sum()
print('Missing values per column:')
print(missing_values)

# Constant columns (columns with only one unique value)
constant_cols = [col for col in df_train.columns if df_train[col].nunique() <= 1]
print(f'Constant columns: {constant_cols}')

Number of duplicate rows: 8
Number of duplicate columns: 0
Missing values per column:
Date of Sale (dd/mm/yyyy)        0
Address                          0
County                           0
Eircode                      37096
Price (€)                        0
Not Full Market Price            0
VAT Exclusive                    0
Description of Property          0
Property Size Description    51225
dtype: int64
Constant columns: []


## Step 3: Helper Functions (Lab Style)


In [8]:
def numeric_summary_table(df, numeric_cols):
    if len(numeric_cols) == 0:
        return pd.DataFrame()
    return df[numeric_cols].describe().T


def categorical_summary_table(df, categorical_cols):
    if len(categorical_cols) == 0:
        return pd.DataFrame()
    return df[categorical_cols].describe().T


def data_quality_overview(df):
    overview = pd.DataFrame({
        'Data Type': df.dtypes.astype(str),
        'Missing Count': df.isna().sum(),
        'Missing %': (df.isna().sum() / len(df) * 100).round(2),
        'Unique Count': df.nunique(dropna=False)
    })
    return overview.sort_values(['Missing %', 'Unique Count'], ascending=[False, False])


def plot_numeric_distributions(df, numeric_cols, pdf_path=None):
    if len(numeric_cols) == 0:
        return

    pp = PdfPages(pdf_path) if pdf_path else None

    for col in numeric_cols:
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))

        clean_series = df[col].dropna()
        axes[0].hist(clean_series, bins=30, color='steelblue', alpha=0.8)
        axes[0].set_title(f'Histogram: {col}')

        axes[1].boxplot(clean_series, vert=False)
        axes[1].set_title(f'Boxplot: {col}')

        plt.tight_layout()
        if pp:
            pp.savefig(fig)
            plt.close(fig)
        else:
            plt.show()

    if pp:
        pp.close()


def plot_categorical_distributions(df, categorical_cols, top_n=20, pdf_path=None):
    if len(categorical_cols) == 0:
        return

    pp = PdfPages(pdf_path) if pdf_path else None

    for col in categorical_cols:
        vc = df[col].astype('object').value_counts(dropna=False)
        vc = vc.head(top_n)

        fig, ax = plt.subplots(figsize=(13, 4))
        vc.plot(kind='bar', ax=ax, color='teal')
        ax.set_title(f'Bar plot (top {top_n}): {col}')
        ax.set_ylabel('Count')
        ax.tick_params(axis='x', rotation=60)
        plt.tight_layout()

        if pp:
            pp.savefig(fig)
            plt.close(fig)
        else:
            plt.show()

    if pp:
        pp.close()


## Step 4: Preliminary Feature Type Conversion for DQR
As in the labs, we first convert obvious fields to useful analysis types before producing DQR tables/plots.


In [9]:
df_train_dqr = df_train.copy()

# Convert date
raw_date_col = 'Date of Sale (dd/mm/yyyy)'
df_train_dqr[raw_date_col] = pd.to_datetime(df_train_dqr[raw_date_col], format='%d/%m/%Y', errors='coerce')

# Convert price to numeric euro values
price_col = 'Price (€)'
df_train_dqr[price_col] = (
    df_train_dqr[price_col]
    .astype(str)
    .str.replace('€', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)
df_train_dqr[price_col] = pd.to_numeric(df_train_dqr[price_col], errors='coerce')

# Convert low-cardinality text fields to category
category_cols = [
    'County',
    'Eircode',
    'Not Full Market Price',
    'VAT Exclusive',
    'Description of Property',
    'Property Size Description'
]

for c in category_cols:
    df_train_dqr[c] = df_train_dqr[c].astype('category')

# Keep high-cardinality Address as object
df_train_dqr['Address'] = df_train_dqr['Address'].astype(str).str.strip()

df_train_dqr.dtypes


Date of Sale (dd/mm/yyyy)    datetime64[us]
Address                                 str
County                             category
Eircode                            category
Price (€)                           float64
Not Full Market Price              category
VAT Exclusive                      category
Description of Property            category
Property Size Description          category
dtype: object

1. Here we are looking at date's column can see that the minimum date is 2016 and the maximum date 2024-12-31 meaning there is no inconsitensies like having really old data or data in the future.

In [15]:
df_train_dqr[raw_date_col].describe()

count                         54000
mean     2020-07-18 18:55:05.600000
min             2016-01-01 00:00:00
25%             2018-04-23 18:00:00
50%             2020-08-11 00:00:00
75%             2022-10-17 00:00:00
max             2024-12-31 00:00:00
Name: Date of Sale (dd/mm/yyyy), dtype: object

2. here we are looking at the price column we can see that there is no negative values zero prices, however 111.280.172 euro seems like a very large outlier and 5,586.84 seems like a very littles price, so more indepth correlation between property type and price should be drawn.

In [16]:
df_train_dqr[price_col].describe()

count        54,000.00
mean        323,719.07
std         950,583.04
min           5,586.84
25%         159,000.00
50%         250,000.00
75%         362,500.00
max     111,280,172.00
Name: Price (€), dtype: float64

In [ ]:
# missing eircode by country to see if any couty has more missing eircode than others maybe used to split: also will compare for missing over time or price range. 
# However, if it is still high missing 69% it has high cardinality and is unlikely to be useful for modeling without significant imputation or feature engineering.
# Most likely we will need to drop this column for modeling. Or reduce it to the first 3 characters and reduce the cardinality.
eircode_missing = (
    df_train_dqr
    .groupby('County')['Eircode']
    .apply(lambda x: x.isna().mean() * 100)
    .sort_values(ascending=False)
)

eircode_missing

County
Kildare     73.46
Donegal     72.25
Meath       71.45
Wicklow     71.24
Monaghan    71.20
Westmeath   71.00
Kerry       70.72
Wexford     70.44
Cavan       70.34
Limerick    70.26
Sligo       69.91
Cork        69.66
Louth       69.62
Offaly      69.61
Mayo        69.60
Roscommon   68.92
Longford    68.57
Galway      68.53
Clare       68.48
Carlow      67.92
Waterford   67.73
Tipperary   66.73
Kilkenny    66.58
Laois       66.44
Leitrim     66.31
Dublin      65.93
Name: Eircode, dtype: float64

In [ ]:
# We can see that from 2021 the missing percentage of eircode has increased significantly, which may be due to changes in data collection or reporting practices. 
# This trend suggests that the eircode data may be less reliable for more recent years, and we should consider this when deciding how to handle missing values in this column for modeling purposes.
# Next we will try to reduce the cardinality of eircode by taking the first 3 characters and see if it can be useful for modeling. And go see it by year and couty.
df_train_dqr['Year'] = df_train_dqr['Date of Sale (dd/mm/yyyy)'].dt.year

df_train_dqr.groupby('Year')['Eircode'].apply(lambda x: x.isna().mean()*100)

Year
2016   99.70
2017   99.77
2018   99.75
2019   99.73
2020   99.40
2021   48.35
2022   23.33
2023   23.73
2024   24.50
Name: Eircode, dtype: float64

In [ ]:
# This is good drops cardinality quite a lot, only 153 unique values. Also, by looking at the largest county dublin, it address include part of the Eircode that we need, the first 3 characters like Dublin 01 etc.
# So we will try to use that and add it to see how many missing values it would be after.
df_train_dqr['Eircode_Routing'] = df_train_dqr['Eircode'].str[:3]
df_train_dqr['Eircode_Routing'].nunique()

153

In [64]:
# After finding all the Key codes area, we can look into addresses and assign them based on the key code area. First we will make a Dictionary.
routing_text = r"""
A92,Ardee,Louth
Y14,Arklow,Wicklow
A84,Ashbourne,Meath
H65,Athenry,Galway
N37,Athlone,Westmeath
R14,Athy,Kildare
K32,Balbriggan,Dublin
F26,Ballina,Mayo
H53,Ballinasloe,Galway
P31,Ballincollig,Cork
F31,Ballinrobe,Mayo
A75,Ballybay,Monaghan
A41,Ballyboughal,Dublin
F35,Ballyhaunis,Mayo
F56,Ballymote,Sligo
P72,Bandon,Cork
P75,Bantry,Cork
H14,Belturbet,Cavan
R42,Birr,Offaly
A94,Blackrock,Dublin
F52,Boyle,Roscommon
A98,Bray,Wicklow
V23,Caherciveen,Kerry
E21,Cahir,Tipperary
R93,Carlow,Carlow
A81,Carrickmacross,Monaghan
N41,Carrick-on-Shannon,Leitrim
E32,Carrick-on-Suir,Tipperary
P43,Carrigaline,Cork
E25,Cashel,Tipperary
F23,Castlebar,Mayo
A75,Castleblaney,Monaghan
F45,Castlerea,Roscommon
H12,Cavan,Cavan
P56,Charleville,Cork
F12,Claremorris,Mayo
H71,Clifden,Galway
P85,Clonakilty,Cork
H23,Clones,Monaghan
E91,Clonmel,Tipperary
P24,Cobh,Cork
H16,Cootehill,Cavan
T12,Cork (centre and southside),Cork
T23,Cork (northside),Cork
P14,Crookstown,Cork
P32,Donoughmore,Cork
P47,Dunmanway,Cork
T56,Watergrasshill,Cork
T34,Whitechurch,Cork
R56,Curragh Camp,Kildare
A63,Delgany,Wicklow
F94,Donegal,Donegal
A92,Drogheda,Louth
D01,Dublin 1,Dublin
D02,Dublin 2,Dublin
D03,Dublin 3,Dublin
D04,Dublin 4,Dublin
D05,Dublin 5,Dublin
D06,Dublin 6,Dublin
D6W,Dublin 6W,Dublin
D07,Dublin 7,Dublin
D08,Dublin 8,Dublin
D09,Dublin 9,Dublin
D10,Dublin 10,Dublin
D11,Dublin 11,Dublin
D12,Dublin 12,Dublin
D13,Dublin 13,Dublin
D14,Dublin 14,Dublin
D15,Dublin 15,Dublin
D16,Dublin 16,Dublin
D17,Dublin 17,Dublin
D18,Dublin 18,Dublin
D20,Dublin 20,Dublin
D22,Dublin 22,Dublin
D24,Dublin 24,Dublin
A86,Dunboyne,Meath
A91,Dundalk,Louth
X35,Dungarvan,Waterford
A85,Dunshaughlin,Meath
R45,Edenderry,Offaly
A83,Enfield,Meath
V95,Ennis,Clare
Y21,Enniscorthy,Wexford
P61,Fermoy,Cork
H91,Galway,Galway
A42,Garristown,Dublin
A96,Glenageary,Dublin
Y25,Gorey,Wexford
A63,Greystones,Wicklow
A82,Kells,Meath
A63,Kilcoole,Wicklow
R51,Kildare,Kildare
R95,Kilkenny,Kilkenny
V93,Killarney,Kerry
X42,Kilmacthomas,Waterford
V35,Kilmallock,Limerick
V15,Kilrush,Clare
A82,Kingscourt,Cavan
P17,Kinsale,Cork
A92,Laytown-Bettystown-Mornington,Meath
F92,Letterkenny,Donegal
F93,Lifford,Donegal
V94,Limerick,Limerick
V31,Listowel,Kerry
T45,Little Island,Cork
N39,Longford,Longford
H62,Loughrea,Galway
K78,Lucan,Dublin
K45,Lusk,Dublin
P12,Macroom,Cork
K36,Malahide,Dublin
P51,Mallow,Cork
W23,Maynooth,Kildare
P25,Midleton,Cork
P67,Mitchelstown,Cork
H18,Monaghan,Monaghan
W34,Monasterevin,Kildare
A94,Monkstown,Dublin
R21,Muine Bheag,Carlow
N91,Mullingar,Westmeath
W91,Naas,Kildare
C15,Navan,Meath
E45,Nenagh,Tipperary
Y34,New Ross,Wexford
W12,Newbridge,Kildare
A63,Newcastle,Wicklow
V42,Newcastle West,Limerick
A63,Newtownmountkennedy,Wicklow
A45,Portlaoise,Laois
A67,Rathnew,Wicklow
A85,Ratoath,Meath
F42,Roscommon,Roscommon
E53,Roscrea,Tipperary
K56,Rush,Dublin
V14,Shannon,Clare
K34,Skerries,Dublin
P81,Skibbereen,Cork
F91,Sligo,Sligo
A83,Summerhill,Meath
K67,Swords,Dublin
E41,Thurles,Tipperary
E34,Tipperary,Tipperary
V92,Tralee,Kerry
H54,Tuam,Galway
R35,Tullamore,Offaly
A82,Virginia,Cavan
X91,Waterford,Waterford
F28,Westport,Mayo
Y35,Wexford,Wexford
A67,Wicklow,Wicklow
P36,Youghal,Cork
"""

import io, re

routing_df = pd.read_csv(io.StringIO(routing_text),
                         sep=',',         # separate by comma
                         engine='python', header=None,
                         names=['Eircode3','Town','County'])
# convert/catch NaNs so the accessor works
routing_df['Town'] = routing_df['Town'].astype('string')      # or .astype(str)
routing_df['Town_up'] = routing_df['Town'].str.upper()

df_train_dqr["Address_clean"] = (
    df_train_dqr["Address"]
    .astype("string")
    .str.upper()
    .str.replace(r"[^A-Z0-9]+", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [65]:
routing_df = routing_df[routing_df['Town'] == routing_df['County']].copy()

routing_df['Town_up'] = (
    routing_df['Town']
    .astype('string')
    .str.upper()
    .str.replace(r"[^A-Z0-9]+", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [68]:
# convenience dicts for the matching loop
town_to_code   = routing_df.set_index('Town_up')['Eircode3'].to_dict()
town_to_county = routing_df.set_index('Town_up')['County'].to_dict()

mask = df_train_dqr['Eircode_Routing_2'].isna()
for town, code in town_to_code.items():
    cnty = town_to_county[town]
    town_re = rf'\b{re.escape(town)}\b'               # match whole word
    hit = df_train_dqr['Address_clean'].str.contains(town_re, na=False)
    county_match = df_train_dqr['County'] == cnty
    df_train_dqr.loc[mask & hit & county_match, 'Eircode_Routing_2'] = code
    mask = df_train_dqr['Eircode_Routing_2'].isna()     # recompute for next iteration

print('routing codes remaining missing:',
      df_train_dqr['Eircode_Routing_2'].isna().mean()*100, '%')

routing codes remaining missing: 25.77777777777778 %


In [67]:
df_train_dqr['Eircode_Routing_2'] = df_train_dqr['Eircode_Routing'].copy()



In [28]:
eircode_year_county = (
    df_train_dqr
    .groupby(['Year', 'County'])['Eircode']
    .apply(lambda x: x.isna().mean()*100)
    .unstack()
)

eircode_year_county

County,Carlow,Cavan,Clare,Cork,Donegal,Dublin,Galway,Kerry,Kildare,Kilkenny,Laois,Leitrim,Limerick,Longford,Louth,Mayo,Meath,Monaghan,Offaly,Roscommon,Sligo,Tipperary,Waterford,Westmeath,Wexford,Wicklow
Year,,,,,,,,,,,,,,,,,,,,,,,,,,
2016,100.00,98.28,99.31,99.42,100.00,99.57,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,99.51,98.11,100.00,100.00,99.07,100.00,100.00,100.00,100.00,100.00
2017,100.00,99.12,98.45,99.68,100.00,99.74,99.37,100.00,100.00,100.00,100.00,100.00,100.00,98.08,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,99.47
2018,100.00,98.94,100.00,99.68,99.36,99.74,100.00,100.00,100.00,100.00,100.00,100.00,99.15,100.00,100.00,100.00,100.00,98.25,98.68,100.00,99.07,100.00,99.47,100.00,100.00,100.00
2019,100.00,100.00,100.00,99.55,98.62,99.73,100.00,100.00,99.42,100.00,100.00,100.00,100.00,100.00,100.00,100.00,99.61,100.00,100.00,100.00,100.00,98.85,99.40,100.00,100.00,100.00
2020,100.00,100.00,100.00,100.00,99.46,99.44,100.00,98.65,98.87,100.00,99.04,96.15,98.38,100.00,99.39,100.00,99.25,97.92,100.00,98.84,98.82,99.43,98.80,99.26,99.14,100.00
2021,54.69,56.12,47.41,48.32,64.67,37.47,53.90,59.12,60.73,54.32,47.22,43.14,47.60,46.55,47.43,60.59,54.44,46.67,53.85,70.91,52.00,45.12,37.91,49.55,48.83,57.62
2022,20.24,23.53,19.71,24.21,21.88,15.30,17.86,26.04,44.11,16.00,20.65,13.95,24.19,11.90,29.38,29.45,37.94,17.39,34.02,14.85,23.40,13.56,28.57,24.77,27.85,33.85
2023,19.48,20.99,18.40,27.13,30.72,16.36,20.78,25.19,40.15,28.00,33.33,25.42,16.06,30.00,29.61,23.17,32.39,23.26,24.00,26.97,19.54,16.36,20.71,27.34,22.45,33.48
2024,26.42,16.28,25.62,32.49,16.18,16.94,19.35,14.74,36.61,30.91,33.10,15.38,22.55,16.67,37.16,20.47,34.98,24.49,24.66,22.47,27.03,20.57,27.98,32.38,27.41,35.48


In [20]:
# check for property missing values by county or price range?
size_missing_county = (
    df_train_dqr
    .groupby('County')['Property Size Description']
    .apply(lambda x: x.isna().mean() * 100)
    .sort_values(ascending=False)
)

size_missing_county

County
Offaly      98.57
Kilkenny    98.39
Westmeath   98.18
Mayo        98.16
Tipperary   97.59
Donegal     97.43
Kerry       97.39
Roscommon   97.17
Waterford   97.14
Leitrim     96.98
Sligo       96.95
Cavan       96.93
Longford    96.84
Wexford     96.59
Clare       96.56
Monaghan    96.37
Galway      96.25
Carlow      95.93
Limerick    95.82
Cork        95.62
Laois       95.56
Wicklow     94.36
Louth       93.58
Kildare     92.92
Dublin      92.82
Meath       90.44
Name: Property Size Description, dtype: float64

## Step 5: Remove Duplicate Rows and Save Cleaned (DQR) Dataset


In [ ]:
before_rows = len(df_train_dqr)
df_train_dqr = df_train_dqr.drop_duplicates().copy()
after_rows = len(df_train_dqr)

print('Rows before duplicate removal:', before_rows)
print('Rows after duplicate removal :', after_rows)
print('Rows removed                 :', before_rows - after_rows)

constant_columns_after = [c for c in df_train_dqr.columns if df_train_dqr[c].nunique(dropna=False) <= 1]
print('Constant columns after duplicate removal:', constant_columns_after)

cleaned_dqr_path = 'ppr-group-22312913-train-cleaned.csv'
df_train_dqr.to_csv(cleaned_dqr_path, index=False)
print('Saved:', cleaned_dqr_path)


## Step 6: Data Quality Report Tables
Numeric and categorical descriptive tables, plus a full overview table.


In [ ]:
numeric_cols = df_train_dqr.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = df_train_dqr.select_dtypes(include=['category', 'object']).columns.tolist()

print('Numeric columns:', numeric_cols)
print('Categorical columns:', categorical_cols)


In [ ]:
dqr_overview = data_quality_overview(df_train_dqr)

numeric_stats = numeric_summary_table(df_train_dqr, numeric_cols)
categorical_stats = categorical_summary_table(df_train_dqr, categorical_cols)

print('Data quality overview:')
display(dqr_overview)

print('Numeric feature summary:')
display(numeric_stats)

print('Categorical feature summary:')
display(categorical_stats)


## Step 7: Data Quality Report Visualisations


In [ ]:
plot_numeric_distributions(
    df_train_dqr,
    numeric_cols,
    pdf_path='ppr-group-22312913-DQR-numeric-plots.pdf'
)

plot_categorical_distributions(
    df_train_dqr,
    categorical_cols,
    top_n=20,
    pdf_path='ppr-group-22312913-DQR-categorical-plots.pdf'
)

print('Saved DQR plots:')
print('- ppr-group-22312913-DQR-numeric-plots.pdf')
print('- ppr-group-22312913-DQR-categorical-plots.pdf')


In [ ]:
monthly_counts = (
    df_train_dqr.set_index('Date of Sale (dd/mm/yyyy)')
    .resample('ME')
    .size()
)

plt.figure(figsize=(12, 4))
monthly_counts.plot(color='darkorange')
plt.title('Monthly Number of Sales (Train Set)')
plt.xlabel('Month')
plt.ylabel('Count')
plt.tight_layout()
plt.show()


## Step 8: DQR Findings (Summary)

Key findings from the training data analysis:
- The raw training set has duplicate rows that should be removed.
- `Eircode` has high missingness (major data completeness issue).
- `Property Size Description` has very high missingness and inconsistent labels (`greater than 125...` vs `greater than or equal to 125...`).
- `Price (€)` and `Date of Sale (dd/mm/yyyy)` are initially stored as text and must be converted to numeric/datetime for analytics.
- `Address` is high-cardinality free text, so it should be retained as text for now and handled carefully in later modeling/feature engineering parts.


## Part 2: Data Quality Plan (DQP)
Create a feature-by-feature quality plan, justify selected actions, apply them, and validate that no missing values remain.


In [ ]:
dqp_plan = pd.DataFrame([
    {
        'Feature': 'Date of Sale (dd/mm/yyyy)',
        'Final Type': 'datetime64[ns]',
        'Issue(s)': 'Stored as string in raw CSV.',
        'Candidate Solutions': 'Keep as text / parse to datetime.',
        'Selected Action': 'Parse using format %d/%m/%Y with errors=coerce.'
    },
    {
        'Feature': 'Address',
        'Final Type': 'object',
        'Issue(s)': 'High-cardinality free-text; possible spacing inconsistencies.',
        'Candidate Solutions': 'Drop / keep raw / basic text normalisation.',
        'Selected Action': 'Keep and apply strip() whitespace cleanup.'
    },
    {
        'Feature': 'County',
        'Final Type': 'category',
        'Issue(s)': 'Categorical text feature.',
        'Candidate Solutions': 'Object / category.',
        'Selected Action': 'Trim spaces and cast to category.'
    },
    {
        'Feature': 'Eircode',
        'Final Type': 'category',
        'Issue(s)': 'Large proportion of missing values.',
        'Candidate Solutions': 'Drop feature / impute mode / impute Unknown.',
        'Selected Action': 'Impute missing values with Unknown and cast to category.'
    },
    {
        'Feature': 'Price (€)',
        'Final Type': 'float64',
        'Issue(s)': 'Currency symbols/commas in text format.',
        'Candidate Solutions': 'Keep string / parse to numeric euro value.',
        'Selected Action': 'Remove symbols and parse to float.'
    },
    {
        'Feature': 'Not Full Market Price',
        'Final Type': 'category',
        'Issue(s)': 'Binary categorical stored as object.',
        'Candidate Solutions': 'Object / category.',
        'Selected Action': 'Standardise text and cast to category.'
    },
    {
        'Feature': 'VAT Exclusive',
        'Final Type': 'category',
        'Issue(s)': 'Binary categorical stored as object.',
        'Candidate Solutions': 'Object / category.',
        'Selected Action': 'Standardise text and cast to category.'
    },
    {
        'Feature': 'Description of Property',
        'Final Type': 'category',
        'Issue(s)': 'Categorical text; rare label variants.',
        'Candidate Solutions': 'Keep as is / map rare values to Other.',
        'Selected Action': 'Keep values, trim spaces, cast to category.'
    },
    {
        'Feature': 'Property Size Description',
        'Final Type': 'category',
        'Issue(s)': 'Very high missingness; inconsistent category wording.',
        'Candidate Solutions': 'Drop feature / impute Unknown + harmonise categories.',
        'Selected Action': 'Impute Unknown; harmonise 125 sq m category labels; cast to category.'
    }
])

display(dqp_plan)


## Step 9: Implement the Data Quality Plan


In [ ]:
df_train_final = df_train_dqr.copy()

text_cols = ['Address', 'County', 'Eircode', 'Not Full Market Price', 'VAT Exclusive',
             'Description of Property', 'Property Size Description']
for col in text_cols:
    df_train_final[col] = df_train_final[col].astype('object')
    df_train_final[col] = df_train_final[col].where(df_train_final[col].notna(), np.nan)
    df_train_final[col] = df_train_final[col].astype('string').str.strip()

for col in ['Eircode', 'Property Size Description']:
    df_train_final[col] = df_train_final[col].fillna('Unknown')

# Harmonise property size labels
size_col = 'Property Size Description'
df_train_final[size_col] = df_train_final[size_col].replace({
    'greater than 125 sq metres': 'greater than or equal to 125 sq metres'
})

# Defensive date parse fallback
date_col = 'Date of Sale (dd/mm/yyyy)'
if not np.issubdtype(df_train_final[date_col].dtype, np.datetime64):
    df_train_final[date_col] = pd.to_datetime(df_train_final[date_col], format='%d/%m/%Y', errors='coerce')

df_train_final['Price (€)'] = pd.to_numeric(df_train_final['Price (€)'], errors='coerce')

if df_train_final['Price (€)'].isna().any():
    df_train_final['Price (€)'] = df_train_final['Price (€)'].fillna(df_train_final['Price (€)'].median())

if df_train_final[date_col].isna().any():
    date_mode = df_train_final[date_col].mode(dropna=True)
    fallback_date = date_mode.iloc[0] if len(date_mode) > 0 else pd.Timestamp('2016-01-01')
    df_train_final[date_col] = df_train_final[date_col].fillna(fallback_date)

for col in ['County', 'Eircode', 'Not Full Market Price', 'VAT Exclusive',
            'Description of Property', 'Property Size Description']:
    df_train_final[col] = df_train_final[col].astype('category')

df_train_final['Address'] = df_train_final['Address'].fillna('Unknown').astype('object')

# Final safety: ensure no NaN values
for col in df_train_final.columns:
    if str(df_train_final[col].dtype).startswith('category'):
        if df_train_final[col].isna().any():
            df_train_final[col] = df_train_final[col].cat.add_categories(['Unknown']).fillna('Unknown')
    elif df_train_final[col].dtype == 'object':
        df_train_final[col] = df_train_final[col].fillna('Unknown')
    elif np.issubdtype(df_train_final[col].dtype, np.number):
        df_train_final[col] = df_train_final[col].fillna(df_train_final[col].median())
    elif np.issubdtype(df_train_final[col].dtype, np.datetime64):
        df_train_final[col] = df_train_final[col].fillna(df_train_final[col].mode(dropna=True).iloc[0])

print('DQP implementation complete.')


## Step 10: Validation After DQP Cleaning


In [ ]:
print('Final shape:', df_train_final.shape)
print('Duplicate rows:', df_train_final.duplicated().sum())

final_missing = df_train_final.isna().sum().sort_values(ascending=False)
print('Missing values by column:')
print(final_missing)
print('Total missing values:', int(final_missing.sum()))

print('Final dtypes:')
print(df_train_final.dtypes)

df_train_final.head()


In [ ]:
print('Final numeric summary:')
display(df_train_final.select_dtypes(include=['int64', 'float64']).describe().T)

print('Final categorical summary:')
display(df_train_final.select_dtypes(include=['category', 'object']).describe().T)


## Step 11: Save Final Clean Dataset for Next Project Parts


In [ ]:
final_path = 'ppr-group-22312913-train-clean-final.csv'

df_export = df_train_final.copy()
df_export['Date of Sale (dd/mm/yyyy)'] = df_export['Date of Sale (dd/mm/yyyy)'].dt.strftime('%Y-%m-%d')

df_export.to_csv(final_path, index=False)
print('Saved:', final_path)


## Deliverables Generated by This Notebook
- `ppr-group-22312913-train-cleaned.csv` (Part 1 cleaned after duplicate removal/type fixes)
- `ppr-group-22312913-train-clean-final.csv` (Part 2 final cleaned dataset, no missing values)
- `ppr-group-22312913-DQR-numeric-plots.pdf`
- `ppr-group-22312913-DQR-categorical-plots.pdf`

Use `ppr-group-22312913-train-clean-final.csv` as input for Parts 3-5.
